# Generate Interleaved Radar Dataset

This notebook creates a new `interleaved datasets` folder that mirrors the project dataset layout: `train`, `val`, and `test`, with one CSV per sequence plus a combined `sequences.csv` summary for each split.

The generator starts from the stable dataset pattern and mixes several radar behaviors into a single interleaved environment.

## Imports and paths

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import yaml

PROJECT_ROOT = Path("..").resolve()
GENERATOR_DIR = PROJECT_ROOT / "synthetic data generation"
DATASET_ROOT = PROJECT_ROOT / "datasets"
OUT_DIR = PROJECT_ROOT / "interleaved datasets"

sys.path.append(str(GENERATOR_DIR))

from radar_generator import Environment, StableRadar, JitterRadar, StaggerRadar, DwellAndSwitchRadar, SlidingRadar

PROJECT_ROOT, GENERATOR_DIR, OUT_DIR


(WindowsPath('C:/Users/MSI/Desktop/Coding Projects/EEE493-494/Bitirme'),
 WindowsPath('C:/Users/MSI/Desktop/Coding Projects/EEE493-494/Bitirme/synthetic data generation'),
 WindowsPath('C:/Users/MSI/Desktop/Coding Projects/EEE493-494/Bitirme/interleaved datasets'))

## Dataset configuration

In [2]:
SEQUENCE_COUNT = 1000
PULSES_PER_SEQUENCE = 128
DURATION_US = 128000.0

TRAIN_COUNT = 700
VAL_COUNT = 100
TEST_COUNT = 200

N = 5 #number of radars in the environment
sampling_resolution = 1 #sampling resolution in microseconds

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

SEQUENCE_COUNT, PULSES_PER_SEQUENCE, DURATION_US


(1000, 128, 128000.0)

## Radar and environment setup

In [3]:

def build_environment(n_radars: int = N):
    env = Environment(N=n_radars, rain=0.0, fog=0.0)
    radar_templates = [
        (StableRadar, {"radar_id": "R_STABLE", "pri": 1000.0, "freq_mean": 9000.0, "prop_distance": 800.0}),
        (JitterRadar, {"radar_id": "R_JITTER", "pri_mean": 1000.0, "pri_jitter_std": 20.0, "freq_mean": 9500.0, "prop_distance": 1200.0}),
        (StaggerRadar, {"radar_id": "R_STAGGER", "stagger_pris": [900.0, 1000.0, 1100.0], "freq_mean": 9800.0, "prop_distance": 1500.0}),
        (DwellAndSwitchRadar, {"radar_id": "R_DWELL", "dwell_pris": [900.0, 1000.0], "pulses_per_dwell": 4, "freq_mean": 9600.0, "prop_distance": 1100.0}),
        (SlidingRadar, {"radar_id": "R_SLIDE", "pri_start": 1100.0, "pri_end": 900.0, "pulses_per_slide": 5, "freq_mean": 9700.0, "prop_distance": 1300.0}),
    ]

    for i in range(max(1, n_radars)):
        cls, base_kwargs = radar_templates[i % len(radar_templates)]
        kwargs = dict(base_kwargs)
        kwargs["radar_id"] = f"{base_kwargs['radar_id']}_{i + 1}"
        env.add_radar(cls(**kwargs))

    return env

build_environment(N)


## Sequence generation

In [4]:
def generate_one_sequence(sequence_id: str) -> pd.DataFrame:
    env = build_environment()
    df = env.generate(duration=DURATION_US, start_time=0.0, noise_rate=0.0)
    if df.empty:
        raise ValueError(f"Generated empty interleaved sequence for {sequence_id}")

    df = df.sort_values("TOA").reset_index(drop=True)
    df["sequence_id"] = sequence_id
    df["pulse_index"] = np.arange(len(df), dtype=int)
    df["radar_type"] = "interleaved"
    df["error_level"] = "basic"

    return df[[
        "Radar_ID", "TOA", "PRI", "Frequency", "PW", "Amplitude", "AOA",
        "Measured_Delta_TOA", "sequence_id", "pulse_index", "radar_type", "error_level"
    ]]

sample_seq = generate_one_sequence("seq_0000")
sample_seq.head()


,Radar_ID,TOA,PRI,Frequency,PW,Amplitude,AOA,Measured_Delta_TOA,sequence_id,pulse_index,radar_type,error_level
0,R_STABLE_1,751.746918,999.999002,9000.0,1.170660,-46.255989,87.548173,0.000000,seq_0000,0,interleaved,basic
1,R_STAGGER_3,976.743543,899.997300,9800.0,0.976365,-53.895767,89.427009,224.996626,seq_0000,1,interleaved,basic
2,R_JITTER_2,1117.525806,1004.148063,9500.0,0.833904,-50.892867,88.424050,140.782263,seq_0000,2,interleaved,basic
3,R_SLIDE_5,1337.534313,1099.994501,9700.0,1.077413,-48.717627,88.746145,220.008507,seq_0000,3,interleaved,basic
4,R_DWELL_4,1660.165190,899.996401,9600.0,1.002131,-44.711313,90.926670,322.630877,seq_0000,4,interleaved,basic


## Save split outputs

## Check for overlapping TOA values before merging datasets

In [5]:

def fix_overlapping_toa(df: pd.DataFrame, sampling_resolution: float = sampling_resolution) -> tuple[pd.DataFrame, int]:
    if df.empty:
        return df.copy(), 0

    out = df.sort_values("TOA").copy().reset_index(drop=True)
    total_overlaps = 0

    while True:
        gaps = np.diff(out["TOA"].to_numpy())
        flagged = np.where(gaps < sampling_resolution)[0]

        if len(flagged) == 0:
            break

        total_overlaps += len(flagged)
        idx = int(flagged[0])
        left_idx = idx
        right_idx = idx + 1
        left_row = out.iloc[left_idx].copy()
        right_row = out.iloc[right_idx].copy()

        if left_row["Amplitude"] >= right_row["Amplitude"]:
            keep_row = left_row
        else:
            keep_row = right_row

        for col in out.columns:
            if col == "TOA":
                continue
            if col in ["Radar_ID", "sequence_id", "radar_type", "error_level"]:
                continue
            values = [left_row[col], right_row[col]]
            if pd.notna(values[0]) and pd.notna(values[1]):
                keep_row[col] = np.random.choice(values)

        out.iloc[left_idx] = keep_row
        out = out.drop(index=right_idx).reset_index(drop=True)

    out = out.sort_values("TOA").reset_index(drop=True)
    return out, total_overlaps

sample_seq_checked, overlap_count = fix_overlapping_toa(sample_seq, sampling_resolution=sampling_resolution)
small_gap_found = False
if len(sample_seq_checked) > 1:
    small_gap_found = np.any(np.diff(sample_seq_checked["TOA"].to_numpy()) < sampling_resolution)

print(f"TOA overlap count before cleanup: {overlap_count}")
if small_gap_found:
    print("Small TOA gaps remain below sampling_resolution.")
else:
    print("No TOA gaps below sampling_resolution remain.")

sample_seq_checked.head()


TOA overlap count before cleanup: 0
No TOA gaps below sampling_resolution remain.


,Radar_ID,TOA,PRI,Frequency,PW,Amplitude,AOA,Measured_Delta_TOA,sequence_id,pulse_index,radar_type,error_level
0,R_STABLE_1,751.746918,999.999002,9000.0,1.170660,-46.255989,87.548173,0.000000,seq_0000,0,interleaved,basic
1,R_STAGGER_3,976.743543,899.997300,9800.0,0.976365,-53.895767,89.427009,224.996626,seq_0000,1,interleaved,basic
2,R_JITTER_2,1117.525806,1004.148063,9500.0,0.833904,-50.892867,88.424050,140.782263,seq_0000,2,interleaved,basic
3,R_SLIDE_5,1337.534313,1099.994501,9700.0,1.077413,-48.717627,88.746145,220.008507,seq_0000,3,interleaved,basic
4,R_DWELL_4,1660.165190,899.996401,9600.0,1.002131,-44.711313,90.926670,322.630877,seq_0000,4,interleaved,basic


In [6]:
def save_split(split_name: str, seq_ids: list[str], root: Path):
    split_dir = root / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    rows = []

    for seq_id in seq_ids:
        seq_df = generate_one_sequence(seq_id)
        seq_df.to_csv(split_dir / f"{seq_id}.csv", index=False)
        rows.append(seq_df)

    combined = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    if not combined.empty:
        combined.to_csv(split_dir / "sequences.csv", index=False)

def write_metadata():
    metadata = {
        "name": "interleaved_basic_1000x128",
        "radar_type": "interleaved",
        "sequence_count": SEQUENCE_COUNT,
        "expected_pulses_per_sequence": PULSES_PER_SEQUENCE,
        "error_level": "basic",
        "split": {
            "train": {"ratio": 0.7, "sequence_count": TRAIN_COUNT, "row_count": TRAIN_COUNT * PULSES_PER_SEQUENCE},
            "val": {"ratio": 0.1, "sequence_count": VAL_COUNT, "row_count": VAL_COUNT * PULSES_PER_SEQUENCE},
            "test": {"ratio": 0.2, "sequence_count": TEST_COUNT, "row_count": TEST_COUNT * PULSES_PER_SEQUENCE},
        },
        "radar_parameters": {
            "radar_ids": ["R_STABLE", "R_JITTER", "R_STAGGER"],
            "pri_us": [1000.0, 1000.0, "staggered"],
            "freq_mean_mhz": [9000.0, 9500.0, 9800.0],
        },
        "environment_parameters": {"duration_us": DURATION_US, "N": 150.0, "rain": 0.0, "fog": 0.0},
        "units": {"TOA": "microseconds", "PRI": "microseconds", "Frequency": "MHz", "PW": "microseconds", "Amplitude": "dBm", "AOA": "degrees"},
    }

    with (OUT_DIR / "metadata.yaml").open("w", encoding="utf-8") as f:
        yaml.safe_dump(metadata, f, sort_keys=False)

OUT_DIR.mkdir(parents=True, exist_ok=True)
train_ids = [f"seq_{i:04d}" for i in range(TRAIN_COUNT)]
val_ids = [f"seq_{i:04d}" for i in range(TRAIN_COUNT, TRAIN_COUNT + VAL_COUNT)]
test_ids = [f"seq_{i:04d}" for i in range(TRAIN_COUNT + VAL_COUNT, SEQUENCE_COUNT)]

save_split("train", train_ids, OUT_DIR)
save_split("val", val_ids, OUT_DIR)
save_split("test", test_ids, OUT_DIR)
write_metadata()

print(f"Created interleaved dataset at: {OUT_DIR}")
print(f"Train: {len(train_ids)} sequences")
print(f"Val: {len(val_ids)} sequences")
print(f"Test: {len(test_ids)} sequences")


Created interleaved dataset at: C:\Users\MSI\Desktop\Coding Projects\EEE493-494\Bitirme\interleaved datasets
Train: 700 sequences
Val: 100 sequences
Test: 200 sequences
